In [ ]:
import pandas as pd
import numpy as np
import re
from ast import literal_eval
from collections import defaultdict
from collections import Counter
import time
import os
from scipy.stats import expon
from scipy.stats import gamma
from scipy.stats import chisquare
from scipy.stats import chi2
from statsmodels.stats.multitest import fdrcorrection

In [ ]:
genes_bed = '/Users/michael/Documents/GitHub/AmpliconRepository/caper/bed_files/hg38_genes.bed'
# genes_bed = '/Users/dhruvkhatri/Downloads/SeniorDesignBafna/AmpliconRepository/caper/bed_files/hg38_genes.bed'

In [ ]:
# Graph class
models = {
	'gamma_random_breakage': [0.3987543483729932, # a
			1.999999999999, # loc
			1749495.5696758535 # scale
			]
}

class Graph:

	def __init__(self, dataset=None, col_format=None, amp_type="ecDNA", loc_type="feature"):
		"""
		Parameters:
			self (Graph) : Graph object 
			dataset (tsv file) : AA aggregated results file
			amp_type (str) : type of focal amplification (ecDNA, BFB, etc.)
			loc_type (str) : cell line or feature
		Return:
			None
		"""
		if dataset is None:
			print("Error: provide Amplicon Architect annotations")	
		else:
			self.amp_type = amp_type
			self.loc_type = loc_type
			self.nodes = []
			self.edges = []
			self.locs = self.ImportLocs(genes_bed)

            
			self.CreateNodes(dataset, col_format)
			self.CreateEdges()

	def CreateNodes(self, dataset, col_format):
		"""
		# OUTDATED DESC
		Create a nodes_df by iterating through the dataset, adding new genes to 
		a list of dictionaries. Update the amplicons for existing genes, then 
		merge the list into the dataframe after processing.
				
		Parameters: 
			self (Graph) : Graph object 
			dataset (tsv file) : AA aggregated results file
		Return: 
			None
		"""
		
		ALL_GENES = 'All genes' if col_format == 'space' else 'All_genes'
		FEATURE_ID = 'Feature ID' if col_format == 'space' else 'Feature_ID'

		# dictionary for fast lookups of gene labels
		gene_index = {}

		# get subset of ecDNA features
		if self.loc_type == "feature":
			all_features = dataset[dataset['Classification'] == self.amp_type]
		# for each feature
		for __, row in all_features.iterrows():
			# get the genes in this feature
			oncogenes = set(self.ExtractGenes(str(row['Oncogenes'])))
			all_genes = self.ExtractGenes(str(row[ALL_GENES]))
			for gene in all_genes:
				feature = row[FEATURE_ID]
				sample = feature.split("_amplicon")[0]
				# if the gene has been seen, add feature and cell line to info
				if gene in gene_index:
					index = gene_index[gene]
					self.nodes[index]['features'].append(feature)
					self.nodes[index]['samples'].append(sample)
				# otherwise add the gene to self.nodes as a new row
				else:
					node_info = {
						'label': gene,
						'oncogene': str(gene in oncogenes),
						'features': [feature],
						'samples': [sample],
						# 'chromosome': self.locs[gene][0],
						# 'start_coord': self.locs[gene][1],
						# 'end_coord': self.locs[gene][2]
                	}
					self.nodes.append(node_info)
					gene_index[gene] = len(self.nodes) - 1
    	
		# remove potential duplicate features or cell lines
		for node_info in self.nodes:
			node_info['features'] = list(set(node_info['features']))
			node_info['samples'] = list(set(node_info['samples']))
	
		# concatenate all nodes as rows in df
		self.nodes_df = pd.DataFrame(self.nodes)
			
	def CreateEdges(self):
		"""
		# OUTDATED DESC
		Create edges by iterating through pairs of nodes in nodes_df based on 
		shared features. Calculate edge properties for each pair, and append 
		the results to the edges_df after processing all pairs.
				
		Parameters: 
			self (Graph) : Graph object 
		Return: 
			None
		"""
		# extract relevant columns
		labels = self.nodes_df['label'].values
		features = self.nodes_df['features'].values
		
		# build reverse index to map features to nodes
		feature_to_nodes = defaultdict(list)
		for index, amps in enumerate(features):
			for amp in amps:
				feature_to_nodes[amp].append(index)

		# collect potential node pairs, avoiding duplicates
		potential_pairs = set()
		for nodelist in feature_to_nodes.values():
			for i, node1 in enumerate(nodelist):
				for node2 in nodelist[i + 1:]:
					potential_pairs.add((node1, node2)) 

		# unzip pairs into source and target nodes
		pairs = list(potential_pairs)
		src_indices, tgt_indices = zip(*pairs)
		
		# retrieve the matching features for all source and target nodes
		src_features = [features[i] for i in src_indices]
		tgt_features = [features[j] for j in tgt_indices]
		
		# remove id information from feature labels
		for feature_list in src_features:
			feature_list = [f.split("_amplicon")[0] for f in feature_list]
		for feature_list in tgt_features:
			feature_list = [f.split("_amplicon")[0] for f in feature_list]

		# calculate all intersections and unions
		inters = [list(set(s) & set(t)) for s,t in zip(src_features, tgt_features)]
		unions = [list(set(s) | set(t)) for s,t in zip(src_features, tgt_features)]

		# filter pairs with non-empty intersections
		non_empty_mask = [len(inter) > 0 for inter in inters]
		src_filtered = [src_indices[i] for i, mask in enumerate(non_empty_mask) if mask]
		tgt_filtered = [tgt_indices[i] for i, mask in enumerate(non_empty_mask) if mask]
		inters_filtered = [inters[i] for i, mask in enumerate(non_empty_mask) if mask]
		unions_filtered = [unions[i] for i, mask in enumerate(non_empty_mask) if mask]

		# calculate weights
		weights = [len(i) / len(u) for i,u in zip(inters_filtered, unions_filtered)]

		# compute p and q values
		p_values = [self.PVal(labels[a], labels[b]) for a,b in zip(src_filtered, tgt_filtered)]
		q_values = self.QVal(p_values)

		# create edges dict and df
		self.edges = [
			{
				'source': labels[i],
				'target': labels[j],
				'weight': weights[idx],
				'inter': inters_filtered[idx],
				'union': unions_filtered[idx],
				'pval': p_values[idx],
				'qval': q_values[idx]
			}
			for idx, (i, j) in enumerate(zip(src_filtered, tgt_filtered))
		]
		self.edges_df = pd.DataFrame(self.edges)

	# helper functions
	# ----------------
	def ExtractGenes(self, input):
		"""
		Parameters: 
			input (str) : ['A', 'B', 'C'] or ["'A'", "'B'", "'C'"]
		Return: 
			list: ["A", "B", "C"]
		"""
		pattern = r"['\"]?([\w./-]+)['\"]?"
		genelist = re.findall(pattern, input)
		return genelist
	
	def ImportLocs(self, bed_file):
		"""
		return a dict of format: {gene (string): (start chromosome (string), 
		start coordinate (int))}
		"""
		gene_coords = pd.read_csv(bed_file, sep="\t", header=None)
		gene_coords['chr_num'] = gene_coords.apply(lambda row: row[0][-1], axis=1)
		locs = {}
		for i,row in gene_coords.iterrows():
			locs[row[3]] = (row['chr_num'], int(row[1]), int(row[2]))
		return locs
	
	def Distance(self, a, b):
		# not in gene dict
		if not a in self.locs or not b in self.locs:
			return 'N/A'
		# multi chr
		if self.locs[a][0] != self.locs[b][0]:
			return 'N/A'
		# single chr
		return abs(self.locs[a][1] - self.locs[b][1])

	def PVal(self, gene_a, gene_b, model='gamma_random_breakage'):
		"""
		generate p_val based on naive random breakage model, considering distance
		but not number of incidents of co-amplification
		"""
		d = self.Distance(gene_a, gene_b)
		if d == 'N/A': return d
		params = models[model]
		cdf = gamma.cdf(d, a=params[0], loc=params[1], scale=params[2])
		return 1-cdf
		# if model == 'expon':
		# 	d = distance(a, b)
		# 	if not type(d) is str:
		# 		cdf = expon.cdf(d, loc=params[0], scale=params[1])
		# 		return 1-cdf
	
	def QVal(self, p_values, alpha=0.05):
		# extract valid p-values
		valid_mask = [p != 'N/A' for p in p_values]
		valid_p_values = [p for p in p_values if p != 'N/A']

		# apply FDR correction only to valid p-values
		_, valid_q_values = fdrcorrection(valid_p_values, alpha=alpha)

		# reconstruct q_values with 'N/A' in the appropriate positions
		valid_q_iter = iter(valid_q_values)
		q_values = [next(valid_q_iter) if valid else -1 for valid in valid_mask]

		return q_values
		# q_values = ['N/A'] * len(p_values)
		# j = 0
		# for i, valid in enumerate(valid_mask):
		# 	if valid:
		# 		q_values[i] = valid_q_values[j]
		# 		j += 1
        
	def Nodes(self):
		try:
			return self.nodes
		except:
			print('Error: build graph')

	def Edges(self):
		try:
			return self.edges
		except:
			print('Error: build graph')
	def Locs(self):
		try:
			return self.locs
		except:
			print('Error: build graph')

In [ ]:
amp_repo_datasets_dir = "/Users/michael/Downloads/amplicon_repo_datasets/"
# amp_repo_datasets_dir = "/Users/dhruvkhatri/Downloads/SeniorDesignBafna/amplicon_repo_datasets/"

In [ ]:
# construct graph
start = time.process_time()

ccle = pd.read_csv(amp_repo_datasets_dir + "ccle_aggregated_results.csv")
graph = Graph(ccle, col_format='space')

end = time.process_time()
print('Construct Graph:', end-start, 's')

# graph.Nodes_df().to_csv('test_nodes.csv', index=False)  
#     graph.Edges_df().to_csv('/Users/michael/Downloads/test_edges_022725.csv', index=False)  

#     print('Test:', graph.Distance('SEPT14', 'ZNF713')) # fails

### Make data and functions accessible in notebook

In [ ]:
total_samples = len(ccle[ccle['Classification']=='ecDNA'])
nodes = graph.Nodes()
edges = graph.Edges()
locs = graph.Locs()

In [ ]:
node_dict = {node['label']: node for node in nodes}

In [ ]:
def Distance(a, b):
    # not in gene dict
    if not a in locs or not b in locs:
        return 'N/A'
    # multi chr
    if locs[a][0] != locs[b][0]:
        return 'N/A'
    # single chr
    return abs(locs[a][1] - locs[b][1])

In [ ]:
def PVal(gene_a, gene_b, model='gamma_random_breakage'):
    """
    generate p_val based on naive random breakage model, considering distance
    but not number of incidents of co-amplification
    """
    d = Distance(gene_a, gene_b)
    if d == 'N/A': return d
    params = models[model]
    cdf = gamma.cdf(d, a=params[0], loc=params[1], scale=params[2])
    return 1-cdf

# Unit Test
Adding an edge to serve as a unit test, can be customized

In [ ]:
def testEdge(name1, name2, samples1, samples2, distance):
    # create nodes and gene distances
    node_dict[name1] = {'label': name1,
                          'oncogene': 'Test',
                          'features': samples1,
                          'samples':  samples1}
    node_dict[name2] = {'label': name2,
                          'oncogene': 'Test',
                          'features': samples2,
                          'samples':  samples2}
    locs[name1] = ('1', 0, 0)
    locs[name2] = ('1', distance, 0)

    # create edge
    inter = list(set(samples1) & set(samples2))
    union = list(set(samples1) | set(samples2))
    return {'source': name1,
            'target': name2,
            'weight': len(inter)/len(union),
            'inter': inter,
            'union': union,
            'pval': -1.0,
            'qval': -1.0}

In [ ]:
# 1. Two genes that are close together and commonly co-amplified

samples1 = list(range(1,21))
samples2 = list(range(1,22))
distance = 200

test1 = testEdge('1A', '1B', samples1, samples2, distance)

In [ ]:
# 2. Two genes that are close together and not commonly co-amplified

samples1 = list(range(1,51))
samples2 = list(range(50,101))
distance = 200

test2 = testEdge('2A', '2B', samples1, samples2, distance)

In [ ]:
# 3. Two genes that are far apart and commonly co-amplified

samples1 = list(range(1,21))
samples2 = list(range(1,22))
distance = 2000000

test3 = testEdge('3A', '3B', samples1, samples2, distance)

In [ ]:
# 4. Two genes that are far apart and are not commonly co-amplified 

samples1 = list(range(1,51))
samples2 = list(range(50,101))
distance = 2000000

test4 = testEdge('4A', '4B', samples1, samples2, distance)

In [ ]:
edges.append(test1)
edges.append(test2)
edges.append(test3)
edges.append(test4)

In [ ]:
test1['weight'], test2['weight'], test3['weight'], test4['weight']

In [ ]:
chi_squared(test1), chi_squared(test2), chi_squared(test3), chi_squared(test4)

In [ ]:
chi_squared_dep(test1), chi_squared_dep(test2), chi_squared_dep(test3), chi_squared_dep(test4)

In [ ]:
# # Unit testing
# test_edge = {'source': 'BORCS8-MEF2B',
#   'target': 'HAPLN4',
#   'weight': 1.0,
#   'inter': ['SKOV3_OVARY_amplicon4_ecDNA_1'],
#   'union': ['SKOV3_OVARY_amplicon4_ecDNA_1'],
#   'pval': 0.6336787070595888,
#   'qval': 0.6800977350972958,}

# edges.append(test_edge)

# Chi-squared calculations
independent, dependent

In [ ]:
# observed counts

# Number of samples with…
# O1,1 = ecDNA co-amplifications of A and B = len(edges[A-B]['inter'])
# O1,2 = ecDNA amplification of A, but not B = len(nodes[A]['samples']) - len(edges[A-B]['inter'])
# O2,1 = ecDNA amplification of B, but not A = len(nodes[B]['samples']) - len(edges[A-B]['inter'])
# O2,2 = ecDNA amplification of neither A or B = total_samples - len(edges[A-B]['inter'])

In [ ]:
def chi_squared(edge):
    pdD = PVal(edge['source'], edge['target'])
    if pdD == 'N/A': return 'N/A', -1
        
    geneA_samples = len(node_dict[edge['source']]['samples'])
    geneB_samples = len(node_dict[edge['target']]['samples'])
    geneAB_samples = len(edge['inter'])
    fA = geneA_samples/total_samples
    fB = geneB_samples/total_samples
    fAB = geneAB_samples/total_samples

    # observed
    O11 = geneAB_samples
    O12 = geneA_samples - geneAB_samples  
    O21 = geneB_samples - geneAB_samples
    O22 = total_samples - geneA_samples - geneB_samples + geneAB_samples
    obs = [O11, O12, O21, O22]
    obs_freq = [o/total_samples for o in obs]
    
    # expected
    E11 = fA * fB * pdD
    E12 = fA * (1 - fB * pdD)
    E21 = fB * (1 - fA * pdD)
    E22 = (1 - fA) * (1 - fB)
    exp = [E11, E12, E21, E22] 
    exp_rescaled = [e/sum(exp) for e in exp]
    
    d = Distance(edge['source'], edge['target'])

    test_statistic = sum([(o-e)*(o-e)/e for o,e in zip(obs_freq, exp_rescaled)])
    cdf = chi2.cdf(test_statistic, df=1)
    p_val = 1-cdf

    # return sum(exp)
    return d, p_val

In [ ]:
node_dict[edges[0]['target']]

In [ ]:
list_sums = []
for i in range(0, len(edges)):
    value = chi_squared(edges[i])
    if value != 'N/A':
        list_sums.append(value)

print(sum(list_sums)/len(list_sums))

In [ ]:
list_sums[:10]

In [ ]:
def chi_squared_dep(edge):
    pdD = PVal(edge['source'], edge['target'])
    if pdD == 'N/A': return 'N/A', -1
        
    geneA_samples = len(node_dict[edge['source']]['samples'])
    geneB_samples = len(node_dict[edge['target']]['samples'])
    geneAB_samples = len(edge['inter'])
    fAandB = geneAB_samples/total_samples
    fA = geneA_samples/total_samples
    fB = geneB_samples/total_samples
    fAorB = fA + fB - fAandB

    # observed
    O11 = geneAB_samples
    O12 = geneA_samples - geneAB_samples
    O21 = geneB_samples - geneAB_samples
    O22 = total_samples - geneA_samples - geneB_samples + geneAB_samples
    obs = [O11, O12, O21, O22]
    obs_freq = [o/total_samples for o in obs]
    
    # expected
    E11 = fAorB * pdD
    E12 = fA * (1 - fB * pdD)
    E21 = fB * (1 - fA * pdD)
    E22 = 1 - fAorB
    exp = [E11, E12, E21, E22] 
    exp_rescaled = [e/sum(exp) for e in exp]

    d = Distance(edge['source'], edge['target'])

    test_statistic = sum([(o-e)*(o-e)/e for o,e in zip(obs_freq, exp_rescaled)])
    cdf = chi2.cdf(test_statistic, df=1)
    p_val = 1-cdf

    # return sum(exp)
    return d, p_val

In [ ]:
list_sums_dep = []
for i in range(0, len(edges)):
    value = chi_squared_dep(edges[i])
    if value != 'N/A':
        list_sums_dep.append(value)

print(sum(list_sums_dep)/len(list_sums_dep))

In [ ]:
list_sums_dep[:10]

In [ ]:
edges[0]

In [ ]:
edges[1]

In [ ]:
chi_squared_ps = []
new_edges = edges
for edge in new_edges:
    d, p_val = chi_squared(edge)
    edge['distance_ind'] = d
    edge['p_val_ind'] = p_val
    chi_squared_ps.append(p_val)

In [ ]:
chi_squared_ps_drop = [p for p in chi_squared_ps if p != -1]
len(chi_squared_ps), len(chi_squared_ps_drop)

In [ ]:
sum(chi_squared_ps_drop)/len(chi_squared_ps_drop)

In [ ]:
max(chi_squared_ps_drop)

In [ ]:
new_edges_df = pd.DataFrame(new_edges)

In [ ]:
new_edges_df = new_edges_df.sort_values('p_val_ind')
# new_edges_df = new_edges_df.drop(['pval', 'qval']

In [ ]:
new_edges_df

In [ ]:
newedges_drop = new_edges_df.drop(new_edges_df[new_edges_df['p_val_ind'] == -1].index)

In [ ]:
newedges_drop

In [ ]:
print(min(newedges_drop['p_val_ind'].to_list()))

In [ ]:
# chi square without assuming independence
chi_squared_dep_ps = []

for edge in new_edges:
    d, p_val = chi_squared_dep(edge)
    edge['distance_dep'] = d
    edge['p_val_dep'] = p_val
    chi_squared_dep_ps.append(p_val)

In [ ]:
chi_squared_dep_ps_drop = [p for p in chi_squared_dep_ps if p != -1]
len(chi_squared_dep_ps), len(chi_squared_dep_ps_drop)

In [ ]:
sum(chi_squared_dep_ps_drop)/len(chi_squared_dep_ps_drop)

In [ ]:
min(chi_squared_dep_ps_drop)

In [ ]:
max(chi_squared_dep_ps_drop)

In [ ]:
new_edges_df = pd.DataFrame(new_edges)

In [ ]:
oncogenes = []

# Open the file and read each line into the list
with open('/Users/dhruvkhatri/Downloads/SeniorDesignBafna/scripts/oncogene_list_hg38.txt', 'r') as f:
    oncogenes = [line.strip() for line in f]  

In [ ]:
def oncogene_number(a, b):
    return int(a in oncogenes) + int(b in oncogenes)

In [ ]:
new_edges_df['oncogene_number'] = new_edges_df.apply(
    lambda row: oncogene_number(row['source'], row['target']), axis=1
)

In [ ]:
sig_edges_df = new_edges_df[(new_edges_df['p_val_ind'] != -1) & (new_edges_df['p_val_ind'] < 1) & (new_edges_df['oncogene_number'] == 2)]

In [ ]:
sig_edges_df.sort_values(['p_val_ind'])

In [ ]:
0:51518/101697
1:8191/15331
2:320/594

In [ ]:
Counter(new_edges_df['oncogene_number'])

In [ ]:
clean_new_edges_df = new_edges_df.drop(new_edges_df[new_edges_df['distance_ind'] == 'N/A'].index)

In [ ]:
clean_new_edges_df = clean_new_edges_df.sort_values(['p_val_ind'])

In [ ]:
new_edges_df = new_edges_df.drop(['pval', 'qval'], axis = 1)
clean_new_edges_df = clean_new_edges_df.drop(['pval', 'qval'], axis = 1)

In [ ]:
clean_new_edges_df.to_csv("/Users/dhruvkhatri/Downloads/SeniorDesignBafna/scripts/clean_chisquared_significance_table.csv")

In [ ]:
# code for histogram
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of p-values
plt.figure(figsize=(8, 5))
sns.histplot(clean_new_edges_df["p_val_dep"], bins=20, kde=True, color="blue")

# Formatting
plt.xlabel("P-Value", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.title("P-Values Assuming Dependence", fontsize=14)
# plt.axvline(x=0.05, color='red', linestyle='dashed', label="Significance Threshold (0.05)")
plt.legend()
plt.show()

In [ ]:
# NOTES FOR INTEGRATION INTO COAMP
# 1. Add total_samples member variable: length of dataset
# 2. For each edge, add xor properties in addition to inter and union: A and not B, B and not A

# Unit Test Histogram 

Highlight the location of the p-value in the distribution where the unit test p-value is

In [ ]:
# Unit test histogram

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of p-values
plt.figure(figsize=(8, 5))
sns.histplot(clean_new_edges_df["p_val_dep"], bins=20, kde=True, color="blue")
threshold_value = clean_new_edges_df.iloc[-1]["p_val_dep"]

# Formatting
plt.xlabel("P-Value", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.title("P-Values Assuming Dependence", fontsize=14)
plt.axvline(x=threshold_value, color='red', linestyle='dashed', label="Unit Test p-value")
plt.legend()
plt.show()